<a href="https://colab.research.google.com/github/Robian-spec/Malaria-Prediction-Model-Using-Machine-Learning/blob/main/Stream_LIt_Malaria_Prediction_using_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install streamlit scikit-learn -q
print("✅ Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 55.7 MB/s eta 0:00:00
✅ Done


In [5]:
%%writefile malaria_app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve)

st.set_page_config(page_title="Malaria Prediction | Group 3", page_icon="🦟", layout="wide")
st.markdown("""
<style>
table {color:#000000 !important; background:#ffffff !important;}
th {background:#1a5276 !important; color:#ffffff !important; padding:10px !important;}
td {color:#000000 !important; background:#ffffff !important; padding:8px !important;}
tr:nth-child(even) td {background:#eaf4fb !important;}
</style>
""", unsafe_allow_html=True)
st.markdown("""
<div style="background:linear-gradient(135deg,#1a5276,#117a65);padding:2rem;border-radius:12px;color:white;margin-bottom:1rem">
<h1>🦟 Malaria Infection Prediction System</h1>
<p>Group 3 · BSc Data Science · Meru University of Science and Technology · 2026<br>
<em>Nyanza · Rift Valley · Central Kenya</em></p>
</div>
""", unsafe_allow_html=True)

st.sidebar.header("📂 Dataset")
uploaded = st.sidebar.file_uploader("Upload CSV", type=["csv"])

@st.cache_data
def load_data(file):
    return pd.read_csv(file)

if not uploaded:
    st.info("⬅️ Upload Final_Malaria_Dataset.csv in the sidebar to begin.")
    st.stop()

df_raw = load_data(uploaded)

@st.cache_data
def preprocess(df):
    df = df.copy()
    for c in ['ID','Health_Facilities','Avg_Income','Disease_Cases','Notes']:
        if c in df.columns:
            df.drop(columns=c, inplace=True)
    df.drop_duplicates(inplace=True)
    df['Region'] = df['Region'].str.strip().str.title()
    df['County'] = df['County'].str.strip().str.title()
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    df[num_cols] = SimpleImputer(strategy='median').fit_transform(df[num_cols])
    for col in ['Rainfall_mm','Temperature_C','Humidity_percent','Malaria_Cases','Lag_1_Month_Cases','Incidence_per_100k']:
        if col in df.columns:
            Q1, Q3 = df[col].quantile([0.25, 0.75])
            df[col] = df[col].clip(Q1 - 1.5*(Q3-Q1), Q3 + 1.5*(Q3-Q1))
    def season(m):
        return ('Long_Rains' if m in [3,4,5] else 'Dry' if m in [6,7,8]
                else 'Short_Rains' if m in [9,10,11] else 'Cool_Dry')
    df['Season']           = df['Month'].apply(season)
    df['Cases_Per_Capita'] = df['Malaria_Cases'] / df['Population'] * 100000
    df['Region_enc']       = LabelEncoder().fit_transform(df['Region'])
    df['County_enc']       = LabelEncoder().fit_transform(df['County'])
    df['Season_enc']       = LabelEncoder().fit_transform(df['Season'])
    F = ['Rainfall_mm','Temperature_C','Humidity_percent','Lag_1_Month_Cases',
         'Incidence_per_100k','Month','Population','Malaria_Cases',
         'Cases_Per_Capita','Region_enc','County_enc','Season_enc']
    return df[F], df['High_Risk_Binary'].astype(int), F, df

X, y, FEATURES, df_proc = preprocess(df_raw)

st.subheader("📊 Dataset Overview")
c1,c2,c3,c4,c5 = st.columns(5)
c1.metric("Records",        f"{len(X):,}")
c2.metric("Features",       len(FEATURES))
c3.metric("Low Risk",       f"{int((y==0).sum()):,}")
c4.metric("High Risk",      f"{int((y==1).sum()):,}")
c5.metric("High-Risk Rate", f"{y.mean()*100:.1f}%")

st.sidebar.header("⚙️ Settings")
test_size  = st.sidebar.slider("Test split %", 10, 40, 20) / 100
models_sel = st.sidebar.multiselect("Models",
    ["Logistic Regression","Random Forest","Gradient Boosting"],
    default=["Logistic Regression","Random Forest","Gradient Boosting"])

if not models_sel:
    st.warning("Select at least one model."); st.stop()

if not st.sidebar.button("🚀 Train Models", type="primary"):
    st.info("👈 Click Train Models in the sidebar to begin."); st.stop()

st.subheader("🔧 Training Models...")
X_tr,X_te,y_tr,y_te = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
sc      = StandardScaler()
X_tr_sc = sc.fit_transform(X_tr)
X_te_sc = sc.transform(X_te)

res    = {}
prog   = st.progress(0)
COLORS = ['#1a5276','#117a65','#d35400']

for i, name in enumerate(models_sel):
    st.write(f"⏳ Training {name}...")
    if name == "Logistic Regression":
        m = LogisticRegression(max_iter=1000, random_state=42)
        m.fit(X_tr_sc, y_tr); yp = m.predict(X_te_sc); ypr = m.predict_proba(X_te_sc)[:,1]
    elif name == "Random Forest":
        m = RandomForestClassifier(n_estimators=200, random_state=42)
        m.fit(X_tr, y_tr); yp = m.predict(X_te); ypr = m.predict_proba(X_te)[:,1]
    else:
        m = GradientBoostingClassifier(n_estimators=200, random_state=42)
        m.fit(X_tr, y_tr); yp = m.predict(X_te); ypr = m.predict_proba(X_te)[:,1]
    res[name] = {
        'model': m,
        'Accuracy':  round(accuracy_score(y_te, yp), 4),
        'Precision': round(precision_score(y_te, yp, zero_division=0), 4),
        'Recall':    round(recall_score(y_te, yp, zero_division=0), 4),
        'F1 Score':  round(f1_score(y_te, yp, zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_te, ypr), 4),
        'yp': yp, 'ypr': ypr, 'cm': confusion_matrix(y_te, yp)
    }
    prog.progress((i+1) / len(models_sel))

st.success("✅ All models trained!")

st.subheader("📈 Performance Summary")
metrics = ['Accuracy','Precision','Recall','F1 Score','ROC-AUC']
best    = max(res, key=lambda n: res[n]['F1 Score'])
rows = ""
for idx, name in enumerate(models_sel):
    ib = name == best
    bg = '#d5f5e3' if ib else ('#eaf4fb' if idx%2==0 else '#ffffff')
    fw = 'bold' if ib else 'normal'
    tag = '  ⭐ Best' if ib else ''
    cells = ''.join(f'<td style="padding:10px;border:1px solid #999;color:#000;text-align:center;">{res[name][m]:.4f}</td>' for m in metrics)
    rows += f'<tr style="background:{bg};"><td style="padding:10px;border:1px solid #999;color:#000;font-weight:{fw};">{name}{tag}</td>{cells}</tr>'
hdrs = ''.join(f'<th style="padding:10px;border:1px solid #999;text-align:center;">{m}</th>' for m in metrics)
st.markdown(f'<table style="width:100%;border-collapse:collapse;font-size:15px;"><tr style="background:#1a5276;color:#fff;"><th style="padding:10px;border:1px solid #999;text-align:left;">Model</th>{hdrs}</tr>{rows}</table>', unsafe_allow_html=True)
st.success(f"🏆 Best Model: **{best}** — F1 = {res[best]['F1 Score']:.4f}")

t1, t2, t3 = st.tabs(["📊 Metrics","🟦 Confusion Matrices","📉 ROC Curves"])
with t1:
    fig, ax = plt.subplots(figsize=(11,5))
    x = np.arange(len(metrics)); w = 0.7/len(models_sel)
    for i, name in enumerate(models_sel):
        vals = [res[name][m] for m in metrics]
        bars = ax.bar(x+i*w-(len(models_sel)-1)*w/2, vals, w, label=name, color=COLORS[i], alpha=0.88, edgecolor='white')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(metrics); ax.set_ylim(0,1.15)
    ax.set_title('Model Performance Comparison', fontweight='bold'); ax.legend()
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout(); st.pyplot(fig); plt.close()
with t2:
    cols = st.columns(len(models_sel))
    for i, name in enumerate(models_sel):
        cm = res[name]['cm']
        fig, ax = plt.subplots(figsize=(4,3.5))
        im = ax.imshow(cm, cmap='Blues'); plt.colorbar(im, ax=ax)
        ax.set_xticks([0,1]); ax.set_yticks([0,1])
        ax.set_xticklabels(['Low','High'], rotation=30); ax.set_yticklabels(['Low','High'])
        thresh = cm.max()/2
        for r in range(2):
            for c in range(2):
                ax.text(c, r, str(cm[r,c]), ha='center', va='center', fontsize=14, fontweight='bold',
                        color='white' if cm[r,c]>thresh else 'black')
        ax.set_title(name, fontweight='bold'); plt.tight_layout(); cols[i].pyplot(fig); plt.close()
with t3:
    fig, ax = plt.subplots(figsize=(8,6))
    for i, name in enumerate(models_sel):
        fpr, tpr, _ = roc_curve(y_te, res[name]['ypr'])
        ax.plot(fpr, tpr, lw=2.5, color=COLORS[i], label=f'{name} (AUC={res[name]["ROC-AUC"]:.3f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves', fontweight='bold'); ax.legend()
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout(); st.pyplot(fig); plt.close()

if "Random Forest" in res:
    st.subheader("🌲 Feature Importance")
    fi = pd.Series(res['Random Forest']['model'].feature_importances_, index=FEATURES).sort_values()
    fig, ax = plt.subplots(figsize=(9,5))
    ax.barh(fi.index, fi.values, color=['#c0392b' if v>=fi.quantile(0.75) else '#1a5276' for v in fi.values], alpha=0.88, edgecolor='white')
    ax.set_title('Feature Importance', fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout(); st.pyplot(fig); plt.close()

st.markdown("---")
st.subheader("🔮 Predict a New Case")
MN = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
st.markdown("#### 🌦️ Climate")
p1,p2,p3 = st.columns(3)
rainfall    = p1.slider("Rainfall (mm)",     0.0, 350.0, 150.0, 5.0)
temperature = p2.slider("Temperature (°C)", 10.0,  40.0,  24.0, 0.5)
humidity    = p3.slider("Humidity (%)",      0.0, 100.0,  70.0, 1.0)
st.markdown("#### 🦟 Cases & Population")
p4,p5,p6 = st.columns(3)
lag_cases  = p4.slider("Lag 1 Month Cases",  0.0, 3500.0, 1200.0, 50.0)
incidence  = p5.slider("Incidence per 100k", 0.0,  600.0,  120.0,  5.0)
month      = p6.selectbox("Month", list(range(1,13)), format_func=lambda m:MN[m-1], index=5)
p7,p8 = st.columns(2)
population = p7.slider("Population",    100000, 3000000, 500000, 10000)
mal_cases  = p8.slider("Malaria Cases",    0.0,  3500.0, 1200.0,  50.0)

if st.button("🔍 Predict Risk", type="primary", use_container_width=True):
    def enc_season(m):
        s = ('Long_Rains' if m in [3,4,5] else 'Dry' if m in [6,7,8]
             else 'Short_Rains' if m in [9,10,11] else 'Cool_Dry')
        return {'Cool_Dry':0,'Dry':1,'Long_Rains':2,'Short_Rains':3}[s], s
    se, sn = enc_season(month)
    cpc = mal_cases / max(population,1) * 100000
    inp = pd.DataFrame([[rainfall,temperature,humidity,lag_cases,incidence,
                         month,population,mal_cases,cpc,0,0,se]], columns=FEATURES)
    st.markdown("### 🎯 Results")
    rcols = st.columns(len(models_sel)); probs = []
    for i, name in enumerate(models_sel):
        X_in = sc.transform(inp) if name=="Logistic Regression" else inp
        pred = res[name]['model'].predict(X_in)[0]
        prob = res[name]['model'].predict_proba(X_in)[0][1]
        probs.append(prob)
        bg = '#c0392b' if pred==1 else '#117a65'
        lbl = "🔴 HIGH RISK" if pred==1 else "🟢 LOW RISK"
        rcols[i].markdown(f'<div style="background:{bg};padding:1.2rem;border-radius:10px;text-align:center;color:white;margin:4px;"><div style="font-size:0.85rem;opacity:0.85;">{name}</div><div style="font-size:1.5rem;font-weight:bold;margin:0.3rem 0;">{lbl}</div><div style="font-size:1rem;">Confidence: {prob*100:.1f}%</div></div>', unsafe_allow_html=True)
    avg = np.mean(probs)
    fig, ax = plt.subplots(figsize=(8,4))
    bars = ax.bar(models_sel, [p*100 for p in probs], color=['#c0392b' if p>=0.5 else '#117a65' for p in probs], edgecolor='white', alpha=0.9, width=0.45)
    ax.axhline(50, color='grey', lw=1.5, linestyle='--', alpha=0.7, label='50% threshold')
    for bar, p in zip(bars, probs):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{p*100:.1f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')
    ax.set_ylim(0,120); ax.set_ylabel('High-Risk Probability (%)'); ax.legend()
    ax.set_title(f'Risk Probability | Ensemble Avg: {avg*100:.1f}%', fontweight='bold')
    ax.spines[['top','right']].set_visible(False); plt.tight_layout(); st.pyplot(fig); plt.close()
    fig, ax = plt.subplots(figsize=(10,1.3))
    ax.barh([0],[1], color='#d5f5e3', height=0.55)
    ax.barh([0],[avg], color='#c0392b' if avg>=0.5 else '#117a65', height=0.55, alpha=0.9)
    ax.axvline(0.5, color='grey', lw=2, linestyle='--'); ax.set_xlim(0,1); ax.set_yticks([])
    ax.set_xticks([0,0.25,0.5,0.75,1.0]); ax.set_xticklabels(['0%','25%','50% threshold','75%','100%'])
    risk_lbl = "🔴 HIGH RISK" if avg>=0.5 else "🟢 LOW RISK"
    ax.set_title(f'Ensemble Risk: {avg*100:.1f}%  →  {risk_lbl}', fontweight='bold')
    ax.spines[['top','right','left']].set_visible(False); plt.tight_layout(); st.pyplot(fig); plt.close()
    st.info(f"📍 Season: **{sn.replace('_',' ')}** | Cases per capita: **{cpc:.1f}** per 100k")

st.markdown("---")
st.caption("Group 3 | BSc Data Science | Meru University of Science and Technology | 2026")


Overwriting malaria_app.py


In [6]:
import subprocess, time, socket, threading, re

PORT = 8501

# Kill any existing streamlit
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Start Streamlit
subprocess.Popen(
    ["streamlit", "run", "malaria_app.py",
     "--server.port", str(PORT),
     "--server.headless", "true",
     "--browser.gatherUsageStats", "false",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=open("st.log","w"), stderr=open("st_err.log","w")
)

print("⏳ Starting Streamlit..."); time.sleep(10)

# Check it started
def port_open(p):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", p)) == 0

if not port_open(PORT):
    print("❌ Streamlit failed:"); print(open("st_err.log").read());
else:
    print("✅ Streamlit running!")
    # Use localhost.run — no account, no token needed
    tunnel = subprocess.Popen(
        ["ssh", "-o", "StrictHostKeyChecking=no",
         "-R", f"80:localhost:{PORT}",
         "nokey@localhost.run"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print("⏳ Opening tunnel..."); time.sleep(8)
    # Read output to find the URL
    import fcntl, os
    fcntl.fcntl(tunnel.stdout, fcntl.F_SETFL, os.O_NONBLOCK)
    output = ""
    for _ in range(15):
        try:
            chunk = tunnel.stdout.read(4096)
            if chunk:
                output += chunk.decode("utf-8", errors="ignore")
        except:
            pass
        urls = re.findall(r'https://[a-zA-Z0-9\-]+\.lhr\.life', output)
        if urls:
            print(f"\n{'='*55}")
            print(f"  🚀  LIVE LINK  →  {urls[0]}")
            print(f"{'='*55}")
            print("  Upload your CSV in the sidebar, then click Train Models.")
            break
        time.sleep(2)
    else:
        # Fallback: try serveo.net
        tunnel.kill()
        tunnel2 = subprocess.Popen(
            ["ssh", "-o", "StrictHostKeyChecking=no",
             "-R", f"80:localhost:{PORT}",
             "serveo.net"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        time.sleep(8)
        fcntl.fcntl(tunnel2.stdout, fcntl.F_SETFL, os.O_NONBLOCK)
        output2 = ""
        for _ in range(10):
            try:
                chunk = tunnel2.stdout.read(4096)
                if chunk:
                    output2 += chunk.decode("utf-8", errors="ignore")
            except:
                pass
            urls2 = re.findall(r'https?://[a-zA-Z0-9\-]+\.serveo\.net', output2)
            if urls2:
                print(f"\n{'='*55}")
                print(f"  🚀  LIVE LINK  →  {urls2[0]}")
                print(f"{'='*55}")
                break
            time.sleep(2)
        else:
            print("Could not get tunnel URL. Raw output:")
            print(output2)


⏳ Starting Streamlit...
✅ Streamlit running!
⏳ Opening tunnel...

  🚀  LIVE LINK  →  https://788ac741071cf8.lhr.life
  Upload your CSV in the sidebar, then click Train Models.
